# Исследование надёжности заёмщиков

## Описание задания

Заказчик — кредитный отдел банка.
Нужно разобраться, влияет ли семейное положение и количество детей клиента на факт погашения кредита в срок.
Входные данные от банка — статистика о платёжеспособности клиентов.

Результаты исследования будут учтены при построении модели кредитного скоринга — специальной системы, которая оценивает способность потенциального заёмщика вернуть кредит банку.

**На что нужно обратить внимание:**

* Как вы описываете найденные в данных проблемы?
* Какие методы замены типов данных, обработки пропусков и дубликатов применяете?
* Категоризируете ли данные?
* Какие выводы делаете?
* Оставляете ли комментарии к шагам?

## Описание данных

* `children` — количество детей в семье
* `days_employed` — общий трудовой стаж в днях
* `dob_years` — возраст клиента в годах
* `education` — уровень образования клиента
* `education_id` — идентификатор уровня образования
* `family_status` — семейное положение
* `family_status_id` — идентификатор семейного положения
* `gender` — пол клиента
* `income_type` — тип занятости
* `debt` — имел ли задолженность по возврату кредитов
* `total_income` — ежемесячный доход
* `purpose` — цель получения кредита

## Основные шаги

### Шаг 1. Откройте таблицу и изучите общую информацию о данных

Задание 1. Импортируйте библиотеку pandas. Считайте данные из csv-файла в датафрейм и сохраните в переменную data. Путь к файлу: https://code.s3.yandex.net/datasets/data.csv.

In [1]:
import pandas as pd
data = pd.read_csv('https://code.s3.yandex.net/datasets/data.csv')

Задание 2. Выведите первые 20 строчек датафрейма data на экран.

In [2]:
data.head(20)

,children,days_employed,dob_years,education,education_id,family_status,family_status_id,gender,income_type,debt,total_income,purpose
0,1,-8437.673028,42,высшее,0,женат / замужем,0,F,сотрудник,0,253875.639453,покупка жилья
1,1,-4024.803754,36,среднее,1,женат / замужем,0,F,сотрудник,0,112080.014102,приобретение автомобиля
2,0,-5623.422610,33,Среднее,1,женат / замужем,0,M,сотрудник,0,145885.952297,покупка жилья
3,3,-4124.747207,32,среднее,1,женат / замужем,0,M,сотрудник,0,267628.550329,дополнительное образование
4,0,340266.072047,53,среднее,1,гражданский брак,1,F,пенсионер,0,158616.077870,сыграть свадьбу
5,0,-926.185831,27,высшее,0,гражданский брак,1,M,компаньон,0,255763.565419,покупка жилья
6,0,-2879.202052,43,высшее,0,женат / замужем,0,F,компаньон,0,240525.971920,операции с жильем
7,0,-152.779569,50,СРЕДНЕЕ,1,женат / замужем,0,M,сотрудник,0,135823.934197,образование
8,2,-6929.865299,35,ВЫСШЕЕ,0,гражданский брак,1,F,сотрудник,0,95856.832424,на проведение свадьбы
9,0,-2188.756445,41,среднее,1,женат / замужем,0,M,сотрудник,0,144425.938277,покупка жилья для семьи


Задание 3. Выведите основную информацию о датафрейме с помощью метода info().

In [3]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21525 entries, 0 to 21524
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   children          21525 non-null  int64  
 1   days_employed     19351 non-null  float64
 2   dob_years         21525 non-null  int64  
 3   education         21525 non-null  object 
 4   education_id      21525 non-null  int64  
 5   family_status     21525 non-null  object 
 6   family_status_id  21525 non-null  int64  
 7   gender            21525 non-null  object 
 8   income_type       21525 non-null  object 
 9   debt              21525 non-null  int64  
 10  total_income      19351 non-null  float64
 11  purpose           21525 non-null  object 
dtypes: float64(2), int64(5), object(5)
memory usage: 2.0+ MB


### Шаг 2. Предобработка данных

Задание 4. Выведите количество пропущенных значений для каждого столбца.
Используйте комбинацию двух методов.

In [4]:
data.isna().sum()

,0
children,0
days_employed,2174
dob_years,0
education,0
education_id,0
family_status,0
family_status_id,0
gender,0
income_type,0
debt,0


In [5]:
data.isnull().sum()

,0
children,0
days_employed,2174
dob_years,0
education,0
education_id,0
family_status,0
family_status_id,0
gender,0
income_type,0
debt,0


Задание 5. В двух столбцах есть пропущенные значения.
Один из них — `days_employed`.
Пропуски в этом столбце вы обработаете на следующем этапе.
Другой столбец с пропущенными значениями — `total_income` — хранит данные о доходах.
На сумму дохода сильнее всего влияет тип занятости, поэтому заполнить пропуски в этом столбце нужно медианным значением по каждому типу из столбца `income_type`.
Например, у человека с типом занятости `сотрудник` пропуск в столбце `total_income` должен быть заполнен медианным доходом среди всех записей с тем же типом.

In [6]:
data['total_income'] = data.groupby('income_type')['total_income'].transform(
    lambda x: x.fillna(x.median()))

In [7]:
data.isna().sum()

,0
children,0
days_employed,2174
dob_years,0
education,0
education_id,0
family_status,0
family_status_id,0
gender,0
income_type,0
debt,0


Задание 6. В данных могут встречаться артефакты (аномалии) — значения, которые не отражают действительность и появились по какой-то ошибке.
Таким артефактом будет отрицательное количество дней трудового стажа в столбце `days_employed`.
Для реальных данных это нормально.
Обработайте значения в этом столбце: замените все отрицательные значения положительными с помощью метода `abs()`.

In [8]:
data['days_employed'] = data['days_employed'].abs()
data.head()

,children,days_employed,dob_years,education,education_id,family_status,family_status_id,gender,income_type,debt,total_income,purpose
0,1,8437.673028,42,высшее,0,женат / замужем,0,F,сотрудник,0,253875.639453,покупка жилья
1,1,4024.803754,36,среднее,1,женат / замужем,0,F,сотрудник,0,112080.014102,приобретение автомобиля
2,0,5623.422610,33,Среднее,1,женат / замужем,0,M,сотрудник,0,145885.952297,покупка жилья
3,3,4124.747207,32,среднее,1,женат / замужем,0,M,сотрудник,0,267628.550329,дополнительное образование
4,0,340266.072047,53,среднее,1,гражданский брак,1,F,пенсионер,0,158616.077870,сыграть свадьбу


Задание 7. Для каждого типа занятости выведите медианное значение трудового стажа в днях из столбца `days_employed`.
У двух типов (безработные и пенсионеры) получатся аномально большие значения.
Исправить такие значения сложно, поэтому оставьте их как есть. Тем более этот столбец не понадобится вам для исследования.

In [9]:
data.groupby('income_type')['days_employed'].median()

,days_employed
income_type,
безработный,366413.652744
в декрете,3296.759962
госслужащий,2689.368353
компаньон,1547.382223
пенсионер,365213.306266
предприниматель,520.848083
сотрудник,1574.202821
студент,578.751554


Задание 8. Выведите перечень уникальных значений столбца `children`.

In [10]:
data['children'].unique()

array([ 1,  0,  3,  2, -1,  4, 20,  5])

Задание 9. В столбце `children` есть два аномальных значения. Удалите строки, в которых встречаются такие аномальные значения из датафрейма `data`.

In [11]:
data = data[data['children'].isin([-1, 20]) == False]

Задание 10. Ещё раз выведите перечень уникальных значений столбца `children`, чтобы убедиться, что артефакты удалены.

In [12]:
data['children'].unique()

array([1, 0, 3, 2, 4, 5])

Задание 11. Заполните пропуски в столбце `days_employed` медианными значениями по каждому типу занятости `income_type`.

In [13]:
data['days_employed'] = data.groupby('income_type')['days_employed'].transform(
    lambda x: x.fillna(x.median()))

Задание 12. Убедитесь, что все пропуски заполнены.
Проверьте себя и ещё раз выведите количество пропущенных значений для каждого столбца с помощью двух методов.

In [14]:
data.isna().sum()

,0
children,0
days_employed,0
dob_years,0
education,0
education_id,0
family_status,0
family_status_id,0
gender,0
income_type,0
debt,0


In [15]:
data.isnull().sum()

,0
children,0
days_employed,0
dob_years,0
education,0
education_id,0
family_status,0
family_status_id,0
gender,0
income_type,0
debt,0


Задание 13. Замените вещественный тип данных в столбце `total_income` на целочисленный с помощью метода `astype()`.

In [16]:
data.dtypes

,0
children,int64
days_employed,float64
dob_years,int64
education,object
education_id,int64
family_status,object
family_status_id,int64
gender,object
income_type,object
debt,int64


In [17]:
data['total_income'] = data['total_income'].astype(int)

In [18]:
data.dtypes

,0
children,int64
days_employed,float64
dob_years,int64
education,object
education_id,int64
family_status,object
family_status_id,int64
gender,object
income_type,object
debt,int64


Задание 14. Обработайте неявные дубликаты в столбце `education`.
В этом столбце есть одни и те же значения, но записанные по-разному: с использованием заглавных и строчных букв.
Приведите их к нижнему регистру.
Проверьте остальные столбцы.

In [19]:
data['education'].unique()

array(['высшее', 'среднее', 'Среднее', 'СРЕДНЕЕ', 'ВЫСШЕЕ',
       'неоконченное высшее', 'начальное', 'Высшее',
       'НЕОКОНЧЕННОЕ ВЫСШЕЕ', 'Неоконченное высшее', 'НАЧАЛЬНОЕ',
       'Начальное', 'Ученая степень', 'УЧЕНАЯ СТЕПЕНЬ', 'ученая степень'],
      dtype=object)

In [20]:
data['education'] = data['education'].str.lower()

In [21]:
data['education'].unique()

array(['высшее', 'среднее', 'неоконченное высшее', 'начальное',
       'ученая степень'], dtype=object)

Задание 15. Выведите на экран количество строк-дубликатов в данных.
Если такие строки присутствуют, удалите их.
Сбрасывать индексы после удаления строк дубликатов с помощью `reset_index(drop=True)` здесь не требуется.

In [22]:
data.duplicated().sum()

np.int64(71)

In [23]:
data = data.drop_duplicates()

Задание 16. На основании диапазонов, указанных ниже, создайте в датафрейме `data` столбец `total_income_category` с категориями:

* 0–30000 — `'E'`;
* 30001–50000 — `'D'`;
* 50001–200000 — `'C'`;
* 200001–1000000 — `'B'`;
* 1000001 и выше — `'A'`.

Например, кредитополучателю с доходом 25000 нужно назначить категорию `'E'`, а клиенту, получающему 235000, — `'B'`.

In [24]:
bins = [0, 30000, 50000, 200000, 1000000, float('inf')]
labels = ['E', 'D', 'C', 'B', 'A']

data['total_income_category'] = pd.cut(
    data['total_income'],
    bins=bins,
    labels=labels,
    right=True)

data.head()

,children,days_employed,dob_years,education,education_id,family_status,family_status_id,gender,income_type,debt,total_income,purpose,total_income_category
0,1,8437.673028,42,высшее,0,женат / замужем,0,F,сотрудник,0,253875,покупка жилья,B
1,1,4024.803754,36,среднее,1,женат / замужем,0,F,сотрудник,0,112080,приобретение автомобиля,C
2,0,5623.422610,33,среднее,1,женат / замужем,0,M,сотрудник,0,145885,покупка жилья,C
3,3,4124.747207,32,среднее,1,женат / замужем,0,M,сотрудник,0,267628,дополнительное образование,B
4,0,340266.072047,53,среднее,1,гражданский брак,1,F,пенсионер,0,158616,сыграть свадьбу,C


Задание 17. Выведите на экран перечень уникальных целей взятия кредита из столбца `purpose`.

In [25]:
data['purpose'].unique()

array(['покупка жилья', 'приобретение автомобиля',
       'дополнительное образование', 'сыграть свадьбу',
       'операции с жильем', 'образование', 'на проведение свадьбы',
       'покупка жилья для семьи', 'покупка недвижимости',
       'покупка коммерческой недвижимости', 'покупка жилой недвижимости',
       'строительство собственной недвижимости', 'недвижимость',
       'строительство недвижимости', 'на покупку подержанного автомобиля',
       'на покупку своего автомобиля',
       'операции с коммерческой недвижимостью',
       'строительство жилой недвижимости', 'жилье',
       'операции со своей недвижимостью', 'автомобили',
       'заняться образованием', 'сделка с подержанным автомобилем',
       'получение образования', 'автомобиль', 'свадьба',
       'получение дополнительного образования', 'покупка своего жилья',
       'операции с недвижимостью', 'получение высшего образования',
       'свой автомобиль', 'сделка с автомобилем',
       'профильное образование', 'высшее об

Задание 18. Создайте функцию, которая на основании данных из столбца `purpose` сформирует новый столбец `purpose_category`, куда войдут следующие категории:

* `'операции с автомобилем'`,
* `'операции с недвижимостью'`,
* `'проведение свадьбы'`,
* `'получение образования'`.

Например, если в столбце `purpose` находится подстрока `'на покупку автомобиля'`, то в столбце `purpose_category` должна появиться строка `'операции с автомобилем'`.

Используйте собственную функцию с именем `categorize_purpose()` и метод `apply()`. Изучите данные в столбце `purpose` и определите, какие подстроки помогут вам правильно определить категорию.

In [26]:
def categorize_purpose(purpose):
    if 'авто' in purpose:
        return 'операции с автомобилем'
    elif 'образован' in purpose:
        return 'получение образования'
    elif 'свадьб' in purpose:
        return 'проведение свадьбы'
    else:
        return 'операции с недвижимостью'

data['purpose_category'] = data['purpose'].apply(categorize_purpose)
data.head()

,children,days_employed,dob_years,education,education_id,family_status,family_status_id,gender,income_type,debt,total_income,purpose,total_income_category,purpose_category
0,1,8437.673028,42,высшее,0,женат / замужем,0,F,сотрудник,0,253875,покупка жилья,B,операции с недвижимостью
1,1,4024.803754,36,среднее,1,женат / замужем,0,F,сотрудник,0,112080,приобретение автомобиля,C,операции с автомобилем
2,0,5623.422610,33,среднее,1,женат / замужем,0,M,сотрудник,0,145885,покупка жилья,C,операции с недвижимостью
3,3,4124.747207,32,среднее,1,женат / замужем,0,M,сотрудник,0,267628,дополнительное образование,B,получение образования
4,0,340266.072047,53,среднее,1,гражданский брак,1,F,пенсионер,0,158616,сыграть свадьбу,C,проведение свадьбы


### Шаг 3. Исследуйте данные и ответьте на вопросы

Ответы на вопросы можно разместить в ячейках тетрадок Jupyter Notebook с типом `markdown`.

* Задание 19. Есть ли зависимость между количеством детей и возвратом кредита в срок?
* Задание 20. Есть ли зависимость между семейным положением и возвратом кредита в срок?
* Задание 21. Есть ли зависимость между уровнем дохода и возвратом кредита в срок?
* Задание 22. Как разные цели кредита влияют на его возврат в срок?
* Задание 23. Приведите возможные причины появления пропусков в исходных данных.
* Задание 24. Объясните, почему заполнить пропуски медианным значением — лучшее решение для количественных переменных.

Ответы сопроводите интерпретацией — поясните, о чём именно говорит полученный вами результат.

In [27]:
data.pivot_table(index='children', values='debt', aggfunc='mean')

,debt
children,
0,0.075438
1,0.092346
2,0.094542
3,0.081818
4,0.097561
5,0.000000


In [28]:
data.pivot_table(index='family_status', values='debt', aggfunc='mean')

,debt
family_status,
Не женат / не замужем,0.097639
в разводе,0.070648
вдовец / вдова,0.066246
гражданский брак,0.093130
женат / замужем,0.075606


In [29]:
data.pivot_table(index='total_income_category', values='debt', aggfunc='mean')

/tmp/ipython-input-2080380337.py:1: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  data.pivot_table(index='total_income_category', values='debt', aggfunc='mean')


,debt
total_income_category,
E,0.090909
D,0.060172
C,0.084982
B,0.070602
A,0.080000


In [30]:
data.pivot_table(index='purpose_category', values='debt', aggfunc='mean')

,debt
purpose_category,
операции с автомобилем,0.093480
операции с недвижимостью,0.072551
получение образования,0.092528
проведение свадьбы,0.079118


Задание 23 - человек мог не работать или не указывать свой доход

Задание 24 - медиана наиболее стабильна (устойчива к выбросам)

### Шаг 4. Напишите общий вывод